# ML Masterclass 06: Model Evaluation & Diagnostics

Welcome to Notebook 06. Anyone can call `.fit()`. A senior engineer proves that the output of `.predict()` is actually generating business value.

In this notebook, we move past simplistic metric definitions and focus on the mathematical and practical diagnostic tools used to evaluate models in production contexts involving heavy class imbalance and temporal dependencies.

---

## 🎓 Socratic Deep Check
Ponder this question before we begin. The answer relies on understanding denominator sensitivity:

> *"If you have a perfectly balanced dataset (50% positive), ROC-AUC and Precision-Recall AUC will look similar. If you have 1% positive and 99% negative (like fraud detection), why does ROC-AUC present a dangerously optimistic view of your model while PR-AUC tells the harsh truth?"*

---

## Table of Contents
1. **Imbalanced Evaluation:** ROC-AUC vs Precision-Recall AUC mathematics.
2. **Probability Calibration:** Why tree models output terrible probabilities and how to fix them (Platt Scaling vs Isotonic Regression).
3. **Validation Strategies:** Why standard K-Fold CV fails completely on Time-Series data.


### 🎓 Junior to Senior: Concept Bridge

**The Senior Perspective:** Juniors report accuracy. Seniors report the metric that MATCHES THE BUSINESS OBJECTIVE. For fraud detection (1% fraud rate), accuracy is meaningless — a model that always predicts 'not fraud' gets 99% accuracy. Use PR-AUC for imbalanced data, calibrated probabilities for threshold-dependent decisions, and time-series splits for temporal data.

**Mental Model:** Think of ROC-AUC vs PR-AUC like this: ROC asks 'how good are you at finding positives WITHOUT bothering too many negatives?' When negatives are 99%, bothering 5% of them (5,000 false alarms) barely moves the FPR needle. PR asks 'of everything you CALLED positive, how many actually were?' — the 5,000 false alarms destroy your precision.

**Common Junior Pitfall:** Using random K-Fold cross-validation on time-series data. This shuffles future data into the training set, creating data leakage from the future. The model appears to predict well because it literally SAW the future. Always use TimeSeriesSplit (walk-forward validation).

---


## 1. Imbalanced Evaluation (ROC vs PR)

When classes are heavily imbalanced (e.g., 99% healthy, 1% cancer), Accuracy is a useless metric. A model that just hardcodes `return 0` achieves 99% accuracy but fails its actual goal completely.

We use AUC (Area Under the Curve) to measure how well the model separates classes across *all* possible decision thresholds.

### 🎓 DEEP QUESTION ANSWERED
**Q:** *Why does ROC-AUC lie on imbalanced data while PR-AUC tells the truth?*

**A:** Look at the denominators.
*   **ROC** plots True Positive Rate vs False Positive Rate ($FPR = \frac{FP}{FP + TN}$).
*   **PR** plots Precision ($Precision = \frac{TP}{TP + FP}$) vs Recall.

In a dataset with 99,000 negatives and 1,000 positives, if your model incorrectly flags 5,000 negatives as positive (Massive False Positives), the $FPR = \frac{5000}{5000 + 94000} = 0.05$. Because the pool of True Negatives is so massive, the FPR barely moves, making the ROC curve look fantastic.

But Precision looks exactly at what we care about: Out of everything you *claimed* was positive, how many actually were? $Precision = \frac{TP}{TP + 5000}$. The massive FP penalty completely destroys the Precision metric, exposing the model's failure.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score
sns.set_theme(style="whitegrid")

# -----------------------------------------------------
# IMPLEMENTATION: Proving ROC Optimism on Imbalanced Data
# -----------------------------------------------------
np.random.seed(42)

# Simulate heavily imbalanced data (9900 Negative, 100 Positive)
y_true = np.array([0]*9900 + [1]*100)

# Simulate a mediocre model that predicts a lot of False Positives but has okay separation
# Negatives get scores centered around 0.1
# Positives get scores centered around 0.6
y_scores_neg = np.random.normal(loc=0.1, scale=0.3, size=9900)
y_scores_pos = np.random.normal(loc=0.6, scale=0.3, size=100)
y_scores = np.clip(np.concatenate([y_scores_neg, y_scores_pos]), 0, 1)

# --- Calculate ROC ---
fpr, tpr, _ = roc_curve(y_true, y_scores)
roc_auc_val = auc(fpr, tpr)

# --- Calculate PR ---
precision, recall, _ = precision_recall_curve(y_true, y_scores)
pr_auc_val = average_precision_score(y_true, y_scores)

# --- Plotting ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.plot(fpr, tpr, color='blue', lw=2, label=f'ROC curve (AUC = {roc_auc_val:.2f})')
ax1.plot([0, 1], [0, 1], color='gray', linestyle='--')
ax1.set_title("ROC Curve (Looks Great!)")
ax1.set_xlabel("False Positive Rate")
ax1.set_ylabel("True Positive Rate")
ax1.legend(loc="lower right")

ax2.plot(recall, precision, color='red', lw=2, label=f'PR curve (AUC = {pr_auc_val:.2f})')
ax2.set_title("Precision-Recall Curve (Reveals the Truth)")
ax2.set_xlabel("Recall")
ax2.set_ylabel("Precision")
ax2.legend(loc="upper right")

plt.tight_layout()
plt.show()

# The ROC AUC is ~0.90! If you show this to your boss, they will deploy the model.
# But the PR AUC is ~0.35. The model is actually terrible.

## 2. Probability Calibration

When `.predict_proba()` returns `0.80`, does that actually mean there is an 80% chance the event will happen?

*   **Logistic Regression** outputs well-calibrated probabilities because it optimizes Log-Loss directly.
*   **Random Forests / Support Vector Machines** DO NOT. Their outputs are just uncalibrated scores between 0 and 1. They push probabilities away from 0 and 1 toward the middle.

If your business logic dictates "Send an email coupon *if* probability of churn > 80%", an uncalibrated Random Forest might literally never output a number higher than `0.75`, meaning you never take action, even if its ranking order (AUC) is perfect.

**Fixing it:** We fit a Calibrator model (like Isotonic Regression or Platt Scaling/Logistic Regression) *on top* of our uncalibrated ML model's outputs using a hold-out validation set.

## 3. Validation Strategies (Time-Series Leakage)

Standard K-Fold Cross Validation shuffles your data and splits it into $K$ chunks.

### 🎓 DEEP QUESTION ANSWERED
**Q:** *Why must time-series data NEVER use standard shuffled K-Fold logic?*

**A:** Because it violates causality and introduces **Data Leakage from the Future**. 
If you are trying to predict stock prices, and Fold 1 uses data from December to train a model that predicts prices in October, the model will inadvertently "learn" information from the future to predict the past. In production, you never have future data. 

**Solution:** Time-Series Split (Rolling Origin Validation). You must train on Jan-Mar, test on April. Then train on Jan-April, test on May. Your model must always walk forward in time.

---
## ✅ Knowledge Check

### Q1: ROC-AUC vs PR-AUC on imbalanced data

<details><summary>👉 Answer</summary>

ROC-AUC uses FPR = FP/(FP+TN) in its X-axis. When TN is massive (99,000), even 5,000 false positives give FPR = 5,000/99,000 = 0.05 — barely moves the curve. PR-AUC uses Precision = TP/(TP+FP). Those same 5,000 FPs give Precision = TP/(TP+5000) — devastating. PR-AUC exposes the truth on imbalanced data; ROC-AUC hides it.
</details>

### Q2: Probability calibration

<details><summary>👉 Answer</summary>

When a Random Forest says predict_proba = 0.80, it does NOT mean 80% chance. RF outputs are just vote proportions (8 of 10 trees voted yes), not calibrated probabilities. If your business rule is 'act when probability > 0.7', an uncalibrated RF might never trigger despite having good ranking. Fix: apply Platt Scaling (logistic regression on outputs) or Isotonic Regression using a held-out calibration set.
</details>

### Q3: Time-series validation

<details><summary>👉 Answer</summary>

Standard K-Fold shuffles data randomly, so Fold 3 might train on December data to predict October — the model 'sees the future.' TimeSeriesSplit enforces temporal order: train on Jan-Mar, test on Apr. Then train on Jan-Apr, test on May. This mimics how the model will actually be used in production where future data never exists.
</details>


---
## 🔨 Practical Practice

### Exercise 1: Create a highly imbalanced dataset (99:1 ratio). Train a model and plot BOTH ROC and PR curves side by side. Show that ROC-AUC ≈ 0.95 while PR-AUC ≈ 0.30. Explain which to report to stakeholders.

### Exercise 2: Demonstrate probability calibration: train a Random Forest, plot its reliability diagram (calibration curve), then apply CalibratedClassifierCV with method='isotonic'. Compare the before/after calibration curves.

### Exercise 3: Implement walk-forward validation from scratch for a time-series dataset. Compare the reported accuracy to standard K-Fold CV. Show that K-Fold overestimates performance due to future data leakage.
